In [ ]:
from pathlib import Path
from tkinter import FALSE
import matplotlib.pyplot as plt
import numpy as np

from snsphd.viz import phd_style
from util.pcr_loader import PcrLoader

colors, swatches = phd_style(jupyterStyle=True)

# ============================================================
# Toggles
# ============================================================
PLOT_ALL = FALSE            # True: all 3 trigger levels (different linestyles per TL); False: hand-picked threshold only
SHOW_DARK = True           # Plot dark-count curves alongside the photon-count curves
USE_MANUAL_SCALE = False   # True: use MANUAL_SCALE_FACTOR; False: auto-normalize at NORMALIZE_BIAS
NORMALIZE_BIAS = 0.090     # µA — each device's avg(3 TLs) at this bias maps to 1.0

# ============================================================
# Datasets
# 35/42 µm: hand-picked single-CSV paths (older format)
# 46/63/87 µm: pick the most-recent PCR CSV in each 4.23.2026 folder
# ============================================================
DATASETS = {
    "35 µm": {
        "path": "../data/DC_Pa250528a_60nm_hero/35um/6_1snap_w0.06_l50_35micron_1.9thermal_start_265_end_XXX.csv",
        "threshold": 0.010,
    },
    "42 µm": {
        "path": "../data/DC_Pa250528a_60nm_hero/42um/6_1snap_w0.06_l50_42um_2.2thermal_start_261_end_XXX.csv",
        "threshold": 0.010,
    },
    "46 µm (QCL)": {
        "folder": "../data/DC_Pa250528a_60nm_hero/4.23.2026/46um",
        "threshold": 0.012,
    },
    "63 µm (QCL)": {
        "folder": "../data/DC_Pa250528a_60nm_hero/4.23.2026/63um",
        "threshold": 0.012,
    },
    "87 µm (QCL)": {
        "folder": "../data/DC_Pa250528a_60nm_hero/4.23.2026/80um",  # folder mislabeled; device is 87 µm
        "threshold": 0.012,
    },
}

PLOT_COLORS = {
    "35 µm": plt.cm.plasma(0.08),
    "42 µm": plt.cm.plasma(0.28),
    "46 µm (QCL)": plt.cm.plasma(0.45),
    "63 µm (QCL)": plt.cm.plasma(0.65),
    "87 µm (QCL)": plt.cm.plasma(0.85),
}

# Cycled across the 3 trigger levels when PLOT_ALL = True
TL_LINESTYLES = ["-", "--", ":"]

# ============================================================
# Manual fudge controls (per device) — leave at defaults unless tweaking
# ============================================================
MANUAL_SCALE_FACTOR = {label: 1.0 for label in DATASETS}
OFFSET_X = {label: 0.0 for label in DATASETS}
OFFSET_Y = {label: 0.0 for label in DATASETS}
CROP_END = {"35 µm": -6, "42 µm": -6, "46 µm (QCL)": -6, "63 µm (QCL)": -6, "87 µm (QCL)": -7}  # int (negative ok) or None

In [ ]:
def latest_pcr_csv(folder):
    folder = Path(folder)
    candidates = [
        p for p in folder.glob("PCR__*.csv")
        if not p.name.startswith("NEGATIVE_PCR") and not p.name.startswith("test_PCR")
    ]
    if not candidates:
        raise FileNotFoundError(f"No PCR CSV in {folder}")
    return max(candidates, key=lambda p: p.stat().st_mtime)


datasets = {}
for label, spec in DATASETS.items():
    path = latest_pcr_csv(spec["folder"]) if "folder" in spec else Path(spec["path"])
    print(f"{label}: {path.name}")
    datasets[label] = PcrLoader.from_threshold_csv(path)

for label, ds in datasets.items():
    print(f"  {label} thresholds: {ds.threshold_map()}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

for label, dataset in datasets.items():
    color = PLOT_COLORS[label]
    spec = DATASETS[label]
    options = dataset.threshold_options()

    # ---------- choose curves to plot ----------
    if PLOT_ALL:
        curves = [
            dataset.get_curve(opt.index, label=f"{label} | {opt.label}")
            for opt in options
        ]
    else:
        curves = [dataset.get_curve(spec["threshold"], label=label)]

    # ---------- per-device normalization scale (only over plotted curves) ----------
    if USE_MANUAL_SCALE:
        scale = MANUAL_SCALE_FACTOR[label]
    else:
        norm_vals = [
            np.interp(NORMALIZE_BIAS, c.bias_current, c.counts) for c in curves
        ]
        norm_value = float(np.mean(norm_vals))
        scale = 1.0 / norm_value if norm_value > 0 else 1.0

    crop = CROP_END[label]
    sl = slice(None, crop) if crop is not None else slice(None)
    ox = OFFSET_X[label]
    oy = OFFSET_Y[label]

    for j, curve in enumerate(curves):
        ls = TL_LINESTYLES[j % len(TL_LINESTYLES)] if PLOT_ALL else "-"
        x = curve.bias_current[sl] + ox
        y = (curve.counts[sl] + oy) * scale
        ax.plot(
            x, y,
            color=color, linewidth=1.7, marker="o", markersize=3,
            linestyle=ls, alpha=0.9, label=curve.label,
        )
        if SHOW_DARK and curve.dark_counts is not None:
            yd = (curve.dark_counts[sl] + oy) * scale
            ax.plot(
                x, yd,
                color=color, linewidth=1.3, marker="x", markersize=3,
                linestyle=ls, alpha=0.45, label=f"DCR {curve.label}",
            )

ax.set_xlim(0.020, 0.12)
ax.set_ylim(-0.05, 1.1)
ax.set_xlabel(r"Bias Current ($\mu$A)")
ax.set_ylabel("Normalized Count Rate")
title_mode = "all 3 TLs" if PLOT_ALL else "hand-picked TL"
title_norm = "manual scale" if USE_MANUAL_SCALE else f"norm at {NORMALIZE_BIAS:.3f} µA"
ax.set_title(f"60 nm hero ({title_mode}, {title_norm})")
ax.legend(frameon=False, fontsize=7, ncol=2, loc="upper left")
ax.grid(alpha=0.2)
plt.tight_layout()

out_dir = Path("../out/60nm_hero")
out_dir.mkdir(parents=True, exist_ok=True)
suffix = "all_TL" if PLOT_ALL else "single_TL"
fig.savefig(out_dir / f"hero_5dev_qcl_pcr_{suffix}.png", dpi=300)
fig.savefig(out_dir / f"hero_5dev_qcl_pcr_{suffix}.pdf", dpi=300)
plt.show()